# Experiment 2 — annotation data, inter-annotator agreement & gold construction

The **human side** of Experiment 2: how the corpus-line sense annotations were
produced, how reliable they are, and how the curated gold standard was built.
This is the data foundation that the LLM evaluation
([`exp2_eval_analysis.ipynb`](exp2_eval_analysis.ipynb)) scores against — the
two notebooks split the work the way Exp 1 split data-design from model-results.

**Scope of this notebook** (deliverable #1):
- §1 annotation pipeline & provenance — the 988 → 839 → 837 funnel and the
  documented ad-hoc decisions (what we excluded and why);
- §2 eval-set composition — a profile of the **837 records actually sent to the
  LLMs** (the final analysis set), with pre-curation figures reported alongside;
- §3 discard & exclusion accounting;
- §4 inter-annotator agreement (annotator ↔ annotator);
- §4b curator ↔ annotator agreement (adjudication reliability);
- §5 raw-annotator `0`-rate vs curated gold `0`-rate (the inventory-gap story);
- §6 novelty candidates (the lexicographic payoff);
- §7 paper-ready tables & figures.

**Data sources** (all CSV — no live Argilla dependency):

| file | what | rows |
|---|---|---|
| `neolatin_concordances/annotations.csv` | raw per-annotator responses | 1,974 (988 × 2) |
| `neolatin_concordances/gold.csv` | resolved gold + curation metadata | 988 sampled |
| `neolatin_concordances/concordance_lines.csv` | **the 837 analyzable records sent to the LLMs** | 837 |
| `senses/senses_l1.csv` | SLS sense inventory (l1) | — |

See [`exp2_pipeline.md`](exp2_pipeline.md) (data pipeline),
[`exp2_analysis_plan.md`](exp2_analysis_plan.md) §§3–6 (analysis design) and
[`../docs/methodology_log.md`](../docs/methodology_log.md) entries `2026-05-24a/b/c`
(the ad-hoc decisions surfaced in §1).

## §0. Setup & data load

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# analysis/ on the path so `import _eval_loader` works regardless of CWD.
HERE = Path.cwd() if (Path.cwd() / "_eval_loader.py").exists() else Path.cwd() / "analysis"
sys.path.insert(0, str(HERE))
import _eval_loader as E
from _eval_loader import (REPO_ROOT, EXP2_EVAL_SET_CSV, EXP2_GOLD_CSV, EXP2_SENSES_CSV,
                          EXP2_LEMMAS, EXP2_TIER_A, EXP2_TIER_B,
                          bootstrap_ci_binary, exp2_lemma_features)
from sklearn.metrics import cohen_kappa_score

ANNOTATIONS_CSV = Path(os.environ.get("DIALEXPOL_DATA_DIR", Path.cwd())) / "neolatin_concordances" / "annotations.csv"
RNG_SEED = 20260516          # shared with the Exp 1 notebooks
BOOTSTRAP_N = 10_000
RNG = np.random.default_rng(RNG_SEED)
TIER = {l: ("A" if l in EXP2_TIER_A else "B") for l in EXP2_LEMMAS}

# Read everything as strings (keep_default_na=False) so blank cells stay "" and
# sense ids / flags are not silently coerced; numeric columns are cast on demand.
gold  = pd.read_csv(EXP2_GOLD_CSV, dtype=str, keep_default_na=False)
evset = pd.read_csv(EXP2_EVAL_SET_CSV, dtype=str, keep_default_na=False)
senses = pd.read_csv(EXP2_SENSES_CSV)
ann = pd.read_csv(ANNOTATIONS_CSV, dtype=str, keep_default_na=False)
for _name, _frame, _required in [
    ('gold.csv', gold, {'lemma','record_id','analyzable','resolution'}),
    ('concordance_lines.csv', evset, {'lemma','record_id','resolution','min_conf','mean_conf'}),
    ('annotations.csv', ann, {'lemma','record_id','annotator','sense_id'}),
]: assert _required <= set(_frame.columns), f'{_name}: missing {sorted(_required - set(_frame.columns))}'

# Composite key (lemma::record_id) — record_id alone is NOT unique across lemmas
# (exp2_pipeline.md "Gotchas"). Every per-record operation keys on this.
for d in (gold, evset, ann):
    d["key"] = d["lemma"] + "::" + d["record_id"]

ANNOTATORS = sorted(ann["annotator"].unique())
print(f"raw responses : {len(ann):4d} rows  ({ann['key'].nunique()} records × {len(ANNOTATORS)} annotators: {ANNOTATORS})")
print(f"gold.csv      : {len(gold):4d} sampled records")
print(f"eval_set.csv  : {len(evset):4d} analyzable records  ← THE SET SENT TO THE LLMs")
print(f"senses (l1)   : {len(senses):4d} sense rows over {senses['lemma'].nunique()} inventory keys")

## §1. Annotation pipeline & provenance

Two annotators (the annotators, the annotators) double-annotated a seeded
988-line sample of 14 lemmas on the `diacorpol` Argilla workspace; each record
carries `match_status` (is the highlighted token really the lemma?),
`quote_quality` (is the passage assessable?), a `sense_id` from the SLS l1
inventory (or `0` = "no listed sense fits"), a free-text `note`, and a 1–5
`confidence`. The curation pipeline
([`exp2_pipeline.md`](exp2_pipeline.md)) resolved these into a gold standard.

**Ad-hoc decisions documented in the methodology log** (these shape what the
analysis set contains and must be reported):

- **`2026-05-24a` — `imperator` `0`-over-assignment.** Annotators assigned `0`
  ("no sense fits") to ~132 `imperator` records that all mean *"Emperor (title)"*
  — a use the SLS inventory covers but lacks a dedicated label for. This is an
  **inventory/guideline gap, not novelty**; the curator reassigned them to
  `imperator.1.1`. We therefore report the **raw-annotator `0`-rate** (a measure
  of inventory friction) **separately** from the **curated gold `0`-rate** (the
  genuine novelty candidates). See §5.
- **`2026-05-24b` — quote/context quality, restoration & empty-context-as-stratum.**
  Candidate records were flagged (`restorable` / `inspect`) by the `context_quality`
  regex (markup / page-number / cut-start / OCR signals); the **restoration actually
  applied was sentence-boundary / segmentation correction** — `restore_fixes.json`
  re-sets the prev/target/next sentence spans so the KWIC unit is the correct
  sentence, **not** OCR/content rewriting. A missing context side (paragraph edge)
  is left **empty by design** — `context_truncated` is an evaluation stratum, not a defect.
- **`2026-05-24c` — confidence strata & high-confidence-disagreement stratum.**
  LLM scoring will be stratified by annotator confidence, and records where the
  two annotators chose *different* senses but **both** at confidence ≥ 4 form a
  distinct "genuine ambiguity" stratum (69 records).
- **Build-gate exclusions (839 → 837).** Two analyzable `comitia` records were
  dropped at eval-set build time: one whose restored sentence lost the `comitia`
  keyword (no placeable `<WSD>` target), and one short dated index/TOC-like
  phrase. Enumerated in §3.
- **`natio` sampling shortfall.** Only 9 `natio` records survived as analyzable
  (a regex/sampling problem flagged in the pipeline) — `natio` figures are thin
  and flagged wherever they appear.

The funnel below quantifies the pipeline.

In [ ]:
# Provenance funnel: counts at each pipeline stage. gold.csv `type` carries the
# curation category; `analyzable` = (final_match_status==valid) & (final_quote_quality==ok).
n_sampled = len(gold)
type_counts = gold["type"].value_counts().to_dict()
n_valid = (gold["final_match_status"] == "valid").sum()
n_ok = (gold["final_quote_quality"] == "ok").sum()
n_analyzable = (gold["analyzable"] == "True").sum()
res = gold[gold["analyzable"] == "True"]["resolution"].value_counts().to_dict()
n_evset = len(evset)
build_drop = sorted(set(gold[gold["analyzable"] == "True"]["key"]) - set(evset["key"]))
ev_resolution = gold[gold["key"].isin(set(evset["key"]))]["resolution"].value_counts().to_dict()
print(f'paper-scope headline (837 eval records): {ev_resolution.get("consensus", 0)} consensus, {ev_resolution.get("adjudicated", 0)} adjudicated')
for _key in build_drop: print('excluded from eval_set:', _key, gold.loc[gold.key == _key, ["resolution", "analyzable", "final_match_status", "final_quote_quality"]].to_dict("records"))

funnel = pd.DataFrame([
    ("sampled (double-annotated)",          n_sampled, "seeded sample, 14 lemmas, 2 annotators"),
    ("  · consensus (annotators agreed)",   type_counts.get("agree", 0), "type=agree → resolution=consensus"),
    ("  · needed adjudication",             sum(type_counts.get(t,0) for t in ("disagree","cleanup","zero","incomplete")),
                                            "disagree+cleanup+zero+incomplete"),
    ("match_status = valid (is the lemma)", n_valid, f"{n_sampled - n_valid} not_lemma (regex cleanup miss)"),
    ("quote_quality = ok (assessable)",     n_ok, f"{n_sampled - n_ok} unassessable (text damage)"),
    ("analyzable (valid AND ok)",           n_analyzable, "the gold standard"),
    ("  · gold = consensus",                res.get("consensus", 0), ""),
    ("  · gold = adjudicated",              res.get("adjudicated", 0), ""),
    ("eval_set.csv (sent to the LLMs)",     n_evset, f"−{n_analyzable - n_evset} build-gate drops (see §3)"),
], columns=["stage", "n", "note"])
print("Annotation → gold → eval-set funnel\n")
print(funnel.to_string(index=False))
print(f"\ncuration categories (gold.csv `type`): {type_counts}")
print(f"build-gate drops (analyzable but not in eval_set): {build_drop}")

## §2. Eval-set composition — what the LLMs saw (the 837 final records)

A profile of the **final analysis set** (`eval_set.csv`), the Exp 2 analog of
`exp1_quote_design`'s composition section. Every covariate here is a
stratification key reused in [`exp2_eval_analysis.ipynb`](exp2_eval_analysis.ipynb).
Pre-curation figures (over the full 988 sample) are reported in §1/§3; this
section is strictly the 837 analyzable records.

In [ ]:
ev = evset.copy()
for c in ("min_conf", "mean_conf", "n_annotators"):
    ev[c] = pd.to_numeric(ev[c], errors="coerce")
ev["is_novelty"] = ev["gold_sense_id"] == "0"
ev["is_adjudicated"] = ev["resolution"] == "adjudicated"
ev["is_ambiguous"] = ev["genuine_ambiguity"] == "True"
ev["is_restored"] = ev["restored"] == "True"
ev["ctx_full"] = ev["context_truncated"] == "none"

feat = exp2_lemma_features()  # n_candidate_senses, n_records, records_per_sense

rows = []
for lemma, g in ev.groupby("lemma"):
    gold_share = g["gold_sense_id"].value_counts(normalize=True).iloc[0]  # dominant-sense share = MFS
    rows.append({
        "tier": TIER[lemma],
        "lemma": lemma,
        "n": len(g),
        "n_cand_senses": int(feat.loc[lemma, "n_candidate_senses"]),
        "n_gold_senses": g["gold_sense_id"].nunique(),
        "mfs_share": round(gold_share, 3),
        "novelty": int(g["is_novelty"].sum()),
        "adjudicated": int(g["is_adjudicated"].sum()),
        "ambiguous_hi": int(g["is_ambiguous"].sum()),
        "restored": int(g["is_restored"].sum()),
        "ctx_full": int(g["ctx_full"].sum()),
        "med_min_conf": g["min_conf"].median(),
        "med_words": int(pd.to_numeric(g["sentence"].str.split().map(len)).median()),
    })
comp = pd.DataFrame(rows).sort_values(["tier", "n"], ascending=[True, False]).reset_index(drop=True)
totals = {"tier": "—", "lemma": "TOTAL", "n": comp["n"].sum(),
          "n_cand_senses": "", "n_gold_senses": "", "mfs_share": "",
          "novelty": comp["novelty"].sum(), "adjudicated": comp["adjudicated"].sum(),
          "ambiguous_hi": comp["ambiguous_hi"].sum(), "restored": comp["restored"].sum(),
          "ctx_full": comp["ctx_full"].sum(), "med_min_conf": "", "med_words": ""}
print("Eval-set composition (the 837 records sent to the LLMs)\n")
print(pd.concat([comp, pd.DataFrame([totals])], ignore_index=True).to_string(index=False))
print("\nNOTE: natio n=9 (sampling shortfall — figures thin). mfs_share is the dominant gold-sense")
print("share = the per-lemma most-frequent-sense baseline reused in exp2_eval_analysis.ipynb.")

In [ ]:
# Composition figures: consensus/adjudicated split, confidence, context strata.
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# (a) per-lemma stacked consensus vs adjudicated
ax = axes[0, 0]
order = comp.sort_values("n", ascending=True)["lemma"].tolist()
cons = ev[ev.resolution == "consensus"].groupby("lemma").size().reindex(order, fill_value=0)
adj  = ev[ev.resolution == "adjudicated"].groupby("lemma").size().reindex(order, fill_value=0)
ax.barh(order, cons, label="consensus", color="#4c72b0")
ax.barh(order, adj, left=cons, label="adjudicated", color="#dd8452")
ax.set_title("(a) records by resolution"); ax.set_xlabel("records"); ax.legend(fontsize=8)

# (b) min-confidence distribution
ax = axes[0, 1]
mc = ev["min_conf"].value_counts().sort_index()
ax.bar(mc.index.astype(int).astype(str), mc.values, color="#55a868")
ax.set_title("(b) min annotator confidence (per record)"); ax.set_xlabel("min(conf_A, conf_B)"); ax.set_ylabel("records")

# (c) context-truncation strata
ax = axes[1, 0]
ct = ev["context_truncated"].value_counts().reindex(["none", "right", "left", "both"], fill_value=0)
ax.bar(ct.index, ct.values, color="#c44e52")
ax.set_title("(c) context availability (kwic_plus_prev stratum)"); ax.set_ylabel("records")

# (d) gold-sense skew: dominant-sense (MFS) share per lemma
ax = axes[1, 1]
ms = comp.sort_values("mfs_share")
ax.barh(ms["lemma"], ms["mfs_share"], color="#8172b3")
ax.axvline(comp["mfs_share"].mean(), ls="--", c="k", lw=1, label=f"macro mean {comp['mfs_share'].mean():.2f}")
ax.set_title("(d) gold-sense skew (MFS share)"); ax.set_xlabel("dominant-sense share"); ax.legend(fontsize=8)

fig.tight_layout(); plt.show()
print("The high MFS share (macro mean ~0.82) is the central difficulty caveat for Exp 2:")
print("the corpus sample is most-frequent-sense-dominated, so accuracy must be read against it.")

## §3. Discard & exclusion accounting

Where the 988 − 837 = 151 non-analysis records went, by reason — the numbers
reviewers ask about. Two signals discard a record: `match_status=not_lemma`
(the highlighted token is not the target lemma — a regex cleanup miss) and
`quote_quality=unassessable` (text too damaged to judge). We also track
**disagreement on the discard signals themselves** (one annotator keeps, the
other discards) and the 2 build-gate drops.

In [ ]:
# Raw per-annotator long form keyed (lemma, record_id, annotator).
raw = ann.copy()
raw["sense_id"] = raw["sense_id"].replace("", np.nan)
raw["confidence"] = pd.to_numeric(raw["confidence"], errors="coerce")

# Per-lemma discard rates (from the FINAL resolved signals in gold.csv).
disc_rows = []
for lemma, g in gold.groupby("lemma"):
    n = len(g)
    not_lemma = (g["final_match_status"] == "not_lemma").sum()
    unassess  = ((g["final_match_status"] == "valid") & (g["final_quote_quality"] == "unassessable")).sum()
    disc_rows.append({"lemma": lemma, "tier": TIER[lemma], "sampled": n,
                      "not_lemma": not_lemma, "unassessable(valid)": unassess,
                      "analyzable": (g["analyzable"] == "True").sum(),
                      "discard_rate": round(1 - (g["analyzable"] == "True").sum() / n, 3)})
disc = pd.DataFrame(disc_rows).sort_values("discard_rate", ascending=False)
print("Per-lemma discard accounting (from gold.csv resolved signals)\n")
print(disc.to_string(index=False))

# Disagreement on the discard signals (annotators split on keep/discard).
match_disagree = gold[gold["ann_match"].str.contains(r"\|") &
                      gold["ann_match"].apply(lambda s: len(set(s.split("|"))) > 1)]
qual_disagree = gold[gold["ann_quality"].str.contains(r"\|") &
                     gold["ann_quality"].apply(lambda s: len(set(s.split("|"))) > 1)]
print(f"\nDisagreement on discard signals (need adjudication on the signal itself):")
print(f"  match_status split (valid vs not_lemma): {len(match_disagree)} records  {dict(match_disagree['ann_match'].value_counts())}")
print(f"  quote_quality split (ok vs unassessable): {len(qual_disagree)} records")

# The 2 build-gate drops (analyzable gold not carried into eval_set).
print(f"\nBuild-gate drops (analyzable → eval_set, 839 → 837):")
for k in sorted(set(gold[gold.analyzable == 'True'].key) - set(evset.key)):
    r = gold[gold.key == k].iloc[0]
    why = "no placeable <WSD> target (restored text lost the keyword)" if r["flag"] == "inspect" else "short dated index/TOC-like phrase (recommend_exclude advisory)"
    print(f"  {k}  (gold {r['gold_sense_id']}) — {why}")

## §4. Inter-annotator agreement (annotator ↔ annotator)

Headline reliability: per-lemma **Cohen's κ** on `sense_id`, computed on the
*analyzable* subset (both annotators marked `valid` + `ok`) over records where
both assigned a sense. Bootstrap 95% CIs (10K resamples over records) since
per-lemma N is modest. Reported per lemma, macro-averaged, and pooled, plus raw
observed agreement. Separate κ on the discard signals verifies annotators agree
on *what to discard*. A confidence-restricted κ (both ≥ 4) is a calibration
sanity check (should exceed the overall κ).

**Reading the three κ figures** (they diverge for principled reasons, and the
paper must report them honestly):
- **Pooled κ** (~0.84) pools every lemma's senses into one confusion matrix.
  Because sense labels are lemma-specific, annotators *never* confuse a `bellum`
  sense with a `civitas` sense — this lemma-blocking inflates pooled κ (an
  optimistic bound).
- **Macro κ** (~0.38) averages per-lemma κ. It is depressed by the **base-rate /
  prevalence paradox**: where one sense dominates (comitia 1.0, imperator 0.95),
  even ~98% raw agreement yields κ≈0 because chance agreement is already
  near-certain (a pessimistic figure on skewed lemmas).
- **Observed agreement** (~0.85) is the robust descriptive number, immune to both
  effects; report it alongside κ. The per-lemma table below shows all three.

In [ ]:
# Build per-record × annotator wide tables from the raw responses.
def _wide(field):
    """Pivot raw responses to (key, lemma) index × annotator columns for `field`,
    restricted to the analyzable subset (both annotators valid & ok)."""
    a = raw.copy()
    # analyzable per record: both annotators valid & ok
    ok = (a.assign(valid=a.match_status.eq("valid"), good=a.quote_quality.eq("ok"))
            .groupby("key")[["valid", "good"]].all())
    keep = ok[(ok["valid"]) & (ok["good"])].index
    a = a[a.key.isin(keep)]
    w = a.pivot_table(index=["lemma", "key"], columns="annotator",
                      values=field, aggfunc="first")
    return w.dropna() if field == "sense_id" else w  # sense: need both present

def _kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0:
        return np.nan, np.nan
    if len(set(a.tolist()) | set(b.tolist())) == 1:
        return np.nan, float((a == b).mean())   # single class → κ undefined
    return float(cohen_kappa_score(a, b)), float((a == b).mean())

def _kappa_ci(a, b, n_boot=BOOTSTRAP_N):
    a, b = np.asarray(a), np.asarray(b)
    n = len(a)
    if n < 5:
        return (np.nan, np.nan)
    ks = []
    for _ in range(n_boot):
        idx = RNG.integers(0, n, n)
        sa, sb = a[idx], b[idx]
        if len(set(sa.tolist()) | set(sb.tolist())) == 1:
            ks.append(1.0 if (sa == sb).all() else 0.0)
        else:
            ks.append(cohen_kappa_score(sa, sb))
    return tuple(np.percentile(ks, [2.5, 97.5]))

A, B = ANNOTATORS[0], ANNOTATORS[1]
wsense = _wide("sense_id")
iaa_rows = []
for lemma, g in wsense.groupby(level="lemma"):
    a, b = g[A].values, g[B].values
    k, agr = _kappa(a, b)
    lo, hi = _kappa_ci(a, b)
    iaa_rows.append({"lemma": lemma, "tier": TIER[lemma], "n_pairs": len(g),
                     "obs_agree": round(agr, 3), "kappa": round(k, 3) if k == k else np.nan,
                     "ci_lo": round(lo, 3) if lo == lo else np.nan,
                     "ci_hi": round(hi, 3) if hi == hi else np.nan})
iaa = pd.DataFrame(iaa_rows).sort_values("kappa")
# pooled κ over all analyzable sense pairs; macro = mean of defined per-lemma κ
pa, pb = wsense[A].values, wsense[B].values
pooled_k, pooled_agr = _kappa(pa, pb)
macro_k = iaa["kappa"].dropna().mean()
print("Per-lemma sense κ (analyzable subset, both annotators assigned a sense)\n")
print(iaa.to_string(index=False))
print(f"\npooled κ = {pooled_k:.3f}  (observed agreement {pooled_agr:.3f}, n={len(wsense)})")
print(f"macro κ  = {macro_k:.3f}  (mean of {iaa['kappa'].notna().sum()} defined per-lemma κ;")
print(f"          single-sense lemmas — e.g. comitia/praefectus — have κ undefined but agreement≈1)")

In [ ]:
# κ on the discard signals (match_status, quote_quality) — over ALL double-annotated
# records (not just analyzable), since these signals decide analyzability.
def _signal_kappa(field):
    w = raw.pivot_table(index=["lemma", "key"], columns="annotator", values=field, aggfunc="first").dropna()
    out = []
    for lemma, g in w.groupby(level="lemma"):
        k, agr = _kappa(g[A].values, g[B].values)
        out.append({"lemma": lemma, "n": len(g), f"{field}_kappa": round(k,3) if k==k else np.nan,
                    f"{field}_agree": round(agr,3)})
    pk, pagr = _kappa(w[A].values, w[B].values)
    return pd.DataFrame(out), pk, pagr

ms_df, ms_k, ms_a = _signal_kappa("match_status")
qq_df, qq_k, qq_a = _signal_kappa("quote_quality")
sig = ms_df.merge(qq_df.drop(columns="n"), on="lemma")
print("κ on the discard signals (do annotators agree on what to discard?)\n")
print(sig.to_string(index=False))
print(f"\npooled match_status κ = {ms_k:.3f} (agree {ms_a:.3f}) | pooled quote_quality κ = {qq_k:.3f} (agree {qq_a:.3f})")
print("match_status: κ is high — annotators strongly agree on what counts as the lemma, so the")
print("  not_lemma discards are reliable, not curation drift.")
print("quote_quality: the prevalence paradox again — raw agreement ~0.89 but κ~0.05, because 'ok'")
print("  dominates (~89%). Read the AGREEMENT here, not κ; the unassessable discards are still consistent.")

In [ ]:
# Confidence-restricted κ: both annotators conf ≥ 4. Should beat overall κ.
wconf = raw.pivot_table(index=["lemma", "key"], columns="annotator", values="confidence", aggfunc="first")
hi_keys = wconf[(wconf[A] >= 4) & (wconf[B] >= 4)].dropna().index.get_level_values("key")
hi = wsense[wsense.index.get_level_values("key").isin(hi_keys)]
k_hi, agr_hi = _kappa(hi[A].values, hi[B].values)
print(f"Confidence-restricted IAA (both annotators conf ≥ 4):")
print(f"  n pairs (hi-conf, analyzable, sense-assigned) = {len(hi)} / {len(wsense)}")
print(f"  pooled κ (hi-conf) = {k_hi:.3f}   vs   pooled κ (all analyzable) = {pooled_k:.3f}")
print(f"  → {'higher, as expected (annotators calibrated to the scale)' if k_hi > pooled_k else 'NOT higher — inspect calibration'}")

In [ ]:
# κ-by-lemma figure with bootstrap CIs (defined-κ lemmas only).
plot = iaa.dropna(subset=["kappa"]).sort_values("kappa")
fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(plot))
ax.errorbar(plot["kappa"], y,
            xerr=[plot["kappa"] - plot["ci_lo"], plot["ci_hi"] - plot["kappa"]],
            fmt="o", color="#4c72b0", ecolor="#aaaaaa", capsize=3)
ax.set_yticks(y); ax.set_yticklabels([f"{l} (n={n})" for l, n in zip(plot["lemma"], plot["n_pairs"])])
for thr, lab in [(0.0, ""), (0.4, "moderate"), (0.6, "substantial"), (0.8, "near-perfect")]:
    ax.axvline(thr, ls=":", c="#cccccc", lw=1)
    if lab: ax.text(thr, len(plot)-0.3, lab, fontsize=7, rotation=90, va="top", color="#888")
ax.axvline(pooled_k, ls="--", c="#c44e52", lw=1.2, label=f"pooled κ = {pooled_k:.2f}")
ax.set_xlabel("Cohen's κ (sense_id, analyzable subset)"); ax.set_title("Inter-annotator agreement by lemma")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()
print("Single-sense lemmas (comitia, praefectus, ...) are omitted: κ is undefined when only one")
print("sense occurs, though raw agreement ≈ 1 (see §4 table obs_agree).")

## §4b. Curator ↔ annotator agreement (adjudication reliability)

The third agreement relationship: on the **adjudicated** records, did the
curator's gold side with annotator A, with B, with both (a record where the
annotators agreed on sense but split on a cleanup signal), or with **neither**
(a curator override / novelty call)? This is the standard adjudicator-reliability
measure for a 2-annotator + tiebreaker design. Computed on analyzable adjudicated
records with a resolved gold sense. Novelty `0` is **not** counted as an override
here — annotators flagged those too (§5/§6).

In [ ]:
# Curator-vs-annotator: compare resolved gold_sense_id to the two annotator sids.
adj = gold[(gold["analyzable"] == "True") & (gold["resolution"] == "adjudicated")
           & (gold["gold_sense_id"].str.strip() != "")].copy()

def _classify(row):
    parts = [p.strip() for p in row["ann_senses"].split("|")]
    parts = [p for p in parts if p != ""]                       # drop not_lemma blanks
    g = row["gold_sense_id"]
    sset = set(parts)
    if len(sset) == 1 and g in sset:
        return "unanimous_annotators"     # agreed on sense; adjudicated on a signal
    if g in sset:
        return "sided_with_one"           # picked one of two disagreeing senses
    return "outside_both"                 # curator override (incl. novelty 0)

adj["cat"] = adj.apply(_classify, axis=1)
cat_counts = adj["cat"].value_counts().reindex(
    ["unanimous_annotators", "sided_with_one", "outside_both"], fill_value=0)
n_override_novelty = ((adj["cat"] == "outside_both") & (adj["gold_sense_id"] == "0")).sum()
n_imp_reassign = ((adj["cat"] == "outside_both") & (adj["lemma"] == "imperator")).sum()
n_other_override = cat_counts["outside_both"] - n_override_novelty - n_imp_reassign
print(f"Curator ↔ annotator on {len(adj)} analyzable adjudicated records:\n")
for k, v in cat_counts.items():
    print(f"  {k:22s}: {v:4d}  ({v/len(adj):.1%})")
print(f"\n  'outside_both' ({cat_counts['outside_both']}) decomposes into:")
print(f"    {n_imp_reassign:3d}  documented imperator inventory-gap reassignment (0/split → imperator.1.1, log 2026-05-24a)")
print(f"    {n_override_novelty:3d}  novelty 0 (annotator-flagged, not a true override)")
print(f"    {n_other_override:3d}  other curator sense-overrides ← the only genuinely discretionary calls")
print("\nThe 'sided_with_one' rate (48%) is the core tiebreaker signal: on a clean sense disagreement")
print("the curator most often endorses one annotator's reading. Consensus golds (n=585) are untouched.")
print("\nPer-lemma breakdown:")
print(adj.groupby("lemma")["cat"].value_counts().unstack(fill_value=0).to_string())

## §5. Raw-annotator `0`-rate vs curated gold `0`-rate

The inventory-gap story (`2026-05-24a`). Annotators assigned `0` ("no sense
fits") far more often than the curated gold retains, because the SLS l1
inventory lacks labels for some real-but-uncatalogued readings — most starkly
`imperator` = "Emperor (title)". Reporting the two rates separately distinguishes
**inventory/guideline friction** (raw rate) from **genuine lexical novelty**
(curated rate, the DiaLexPoL payoff in §6).

In [ ]:
# Raw 0-rate: per lemma, fraction of submitted responses with sense_id == "0"
# (over responses where the annotator assigned a sense at all — i.e. not not_lemma).
raw_sense = raw[raw["match_status"] == "valid"].copy()
raw0 = raw_sense.assign(is0=raw_sense["sense_id"].fillna("") == "0").groupby("lemma")["is0"].agg(["sum", "count"])
raw0["raw_0_rate"] = (raw0["sum"] / raw0["count"]).round(3)

# Curated gold 0-rate: per lemma over the analyzable eval_set.
gold0 = ev.assign(is0=ev["gold_sense_id"] == "0").groupby("lemma")["is0"].agg(["sum", "count"])
gold0["gold_0_rate"] = (gold0["sum"] / gold0["count"]).round(3)

cmp0 = pd.DataFrame({
    "raw_0_responses": raw0["sum"].astype(int),
    "raw_0_rate": raw0["raw_0_rate"],
    "gold_0_records": gold0["sum"].reindex(raw0.index).fillna(0).astype(int),
    "gold_0_rate": gold0["gold_0_rate"].reindex(raw0.index).fillna(0),
}).sort_values("raw_0_rate", ascending=False)
print("Raw-annotator 0-rate vs curated gold 0-rate (per lemma)\n")
print(cmp0.to_string())
print(f"\nTOTAL raw 0-responses: {int(raw0['sum'].sum())}   →   curated gold 0-records: {int(gold0['sum'].sum())}")
print("imperator is the inventory-gap signal: a high raw rate that collapses to 0 after the")
print("curator reassigned 'Emperor (title)' to imperator.1.1 (methodology log 2026-05-24a).")

## §6. Novelty candidates — the lexicographic payoff

The curated gold `0` records are the **genuine novelty candidates**: corpus uses
the SLS inventory does not cover, the cleanest research output of the experiment
and the input to DiaLexPoL candidate-sense compilation. Listed with the passage
and the annotators' notes; the `proposed_category` column is a manual
categorization template (novelty / inventory-gap / insufficient-context).

In [ ]:
nov = ev[ev["gold_sense_id"] == "0"][["key", "lemma", "siglum", "sentence"]].copy()
# attach annotator notes from raw responses
notes = (raw.groupby("key")["note"].apply(lambda s: " || ".join(n for n in s if str(n).strip()))
            .rename("annotator_notes"))
nov = nov.merge(notes, on="key", how="left")
nov["proposed_category"] = ""   # manual: novelty | inventory_gap | insufficient_context
nov["sentence"] = nov["sentence"].str.slice(0, 200)
print(f"Curated novelty candidates (gold = 0): {len(nov)} records across {nov['lemma'].nunique()} lemma(s)\n")
with pd.option_context("display.max_colwidth", 200, "display.width", 220):
    print(nov[["lemma", "key", "sentence", "annotator_notes"]].to_string(index=False))
# The private workflow worklist is not part of the reproducibility release.
# Keep the candidates available in-memory for inspection instead of writing
# to the old working-tree path.
display(nov)

## §7. Paper-ready outputs

The annotation/reliability table reviewers expect: per-lemma N, sources, sense
count, IAA κ, adjudication and novelty rates. Exported to LaTeX. The κ-by-lemma
figure (§4) and composition figures (§2) are the accompanying visuals.

In [ ]:
paper = comp.merge(iaa[["lemma", "obs_agree", "kappa"]], on="lemma", how="left")
paper["n_sources"] = paper["lemma"].map(ev.groupby("lemma")["siglum"].nunique())
paper["%adjud"] = (100 * paper["adjudicated"] / paper["n"]).round(1)
paper_tbl = paper[["tier", "lemma", "n", "n_sources", "n_cand_senses", "mfs_share",
                   "kappa", "obs_agree", "%adjud", "novelty"]].rename(columns={
    "n_cand_senses": "L1 senses", "mfs_share": "MFS", "obs_agree": "agree", "kappa": "κ"})
print("Methodology table (paper-ready)\n")
print(paper_tbl.to_string(index=False))

latex = paper_tbl.to_latex(index=False, na_rep="—", float_format="%.2f",
    caption="Experiment 2 annotation summary: per-lemma sample size, sources, L1 sense count, "
            "most-frequent-sense share, inter-annotator agreement (Cohen's $\\kappa$ and raw agreement), "
            "adjudication rate, and curated novelty count.",
    label="tab:exp2-annotation")
OUTPUT_DIR = Path(os.environ.get('DIALEXPOL_OUTPUT_DIR', Path.cwd() / 'outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'exp2_annotation_table.tex').write_text(latex, encoding='utf-8')
display(paper_tbl)
print("\nHeadline numbers for the methods section:")
print(f"  · {len(evset)} analyzable records over 14 lemmas, double-annotated (2 annotators)")
print(f"  · pooled sense κ = {pooled_k:.3f}; macro κ = {macro_k:.3f}; pooled observed agreement = {pooled_agr:.3f}")
print(f"  · {(ev['resolution']=='adjudicated').sum()} adjudicated ({(ev['resolution']=='adjudicated').mean():.1%}); "
      f"{ev['is_ambiguous'].sum()} high-confidence-disagreement records")
print(f"  · {ev['is_novelty'].sum()} curated novelty candidates; macro MFS share {comp['mfs_share'].mean():.3f}")

---

### What this notebook feeds into

- **`exp2_eval_analysis.ipynb`** (LLM results) scores the 5-provider CORE run
  against this gold and uses every covariate profiled in §2 as a stratum:
  the genuine-ambiguity set (§2/§4b), the confidence strata (§2/§4),
  `context_truncated` (§2), and the MFS baseline (§2 `mfs_share`,
  `exp2_mfs_baseline()` in `_eval_loader.py`).
- The **raw vs curated `0`-rate** (§5) and **novelty candidates** (§6) are the
  human-only findings; the LLM `0`-assignment behaviour (under-abstention) is
  analysed against them in the eval notebook.

### Cross-references
- [`exp2_pipeline.md`](exp2_pipeline.md) — the data pipeline that produced these CSVs.
- [`exp2_analysis_plan.md`](exp2_analysis_plan.md) §§3–6 — the analysis design this implements.
- [`../docs/methodology_log.md`](../docs/methodology_log.md) `2026-05-24a/b/c` — the ad-hoc decisions.
- [`exp2_runs.md`](exp2_runs.md) — the LLM run manifest.